# `Memory in Langchain`
---

# Memory in LangChain

Memory is one of the most important concepts when building **chatbots, conversational RAG systems, and AI agents**.

The key idea is:

> **An LLM does not automatically remember previous conversations.**
> If you want conversational memory, your application must store the messages and provide the relevant history back to the model.

---

## 1. What is Memory in LangChain?

**Memory** is the mechanism used by a LangChain application to **store, retrieve, and reuse previous conversation messages**.

For example:

```text
User: My name is Arun.
AI: Nice to meet you, Arun.

User: What is my name?
AI: Your name is Arun.
```

For the second question, the LLM needs access to:

```text
User: My name is Arun.
```

That previous message is the **chat history**.

So conceptually:

```text
User
  ↓
Application
  ↓
Chat History / Memory
  ↓
LLM
  ↓
Response
```

---

# 2. Why Do We Need Memory?

Without memory, every request can be treated as an independent request.

Suppose you send:

### Request 1

```text
User:
My name is Arun.
```

The model receives:

```text
"My name is Arun."
```

and responds.

Then you send:

### Request 2

```text
User:
What is my name?
```

If you only send:

```text
"What is my name?"
```

the model has no information saying that your name is Arun.

It may respond:

```text
"I don't know your name."
```

But if your application sends the previous conversation as context:

```text
Human: My name is Arun.
AI: Nice to meet you, Arun.

Human: What is my name?
```

the model can answer:

```text
Your name is Arun.
```

---

# 3. Why Doesn't an LLM Remember Messages?

This is a very important concept.

An LLM is fundamentally a **stateless inference system** from the application's perspective.

Consider:

```python
response = llm.invoke("My name is Arun.")
```

The model processes the input and generates an output.

Later:

```python
response = llm.invoke("What is my name?")
```

The second invocation does **not automatically contain the first invocation**.

Conceptually:

```text
Call 1
"My name is Arun."
       ↓
      LLM
       ↓
"Nice to meet you."
```

Then:

```text
Call 2
"What is my name?"
       ↓
      LLM
       ↓
??? 
```

The LLM isn't automatically maintaining a database of your previous requests.

---

# 4. Important Distinction: Context vs Memory

These terms are often confused.

### Context

Information currently provided to the model.

```text
System message
+
Previous messages
+
Current user message
```

### Memory

The **mechanism used by your application to preserve and retrieve information** across interactions.

So:

```text
Memory
   ↓
retrieves history
   ↓
History becomes context
   ↓
LLM
```

A useful mental model:

> **Memory stores the conversation; context is what the LLM actually receives.**

---

# 5. Does ChatGPT Have Memory?

A conversational application can have memory, but the important architectural point is:

```text
Application
     ↓
Memory / Database
     ↓
Retrieve relevant information
     ↓
Prompt / Messages
     ↓
LLM
```

The LLM itself doesn't magically query your application's database.

The application is responsible for supplying the information.

---

# 6. LangChain Chat History

LangChain provides abstractions for managing messages.

A typical history contains messages such as:

```text
HumanMessage
AIMessage
SystemMessage
ToolMessage
```

Example:

```python
[
    HumanMessage(content="My name is Arun"),
    AIMessage(content="Nice to meet you, Arun"),
    HumanMessage(content="What is my name?")
]
```

The history can then be passed to the model.

---

# 7. Classes for Managing Chat History

The classes you mentioned can be understood in two broad categories:

### Chat-history storage

```text
BaseChatMessageHistory
       │
       ├── ChatMessageHistory
       ├── RedisChatMessageHistory
       ├── FirestoreChatMessageHistory
       └── StreamlitChatMessageHistory
```

### Conversation-aware runnable

```text
RunnableWithMessageHistory
```

`RunnableWithMessageHistory` sits one level above the history implementations.

---

# 8. BaseChatMessageHistory

## What is it?

`BaseChatMessageHistory` is the **base abstraction/interface** for chat history in LangChain.

It defines the operations that a chat-history implementation should provide.

Conceptually:

```text
BaseChatMessageHistory
        ↓
defines interface
        ↓
add messages
get messages
clear messages
```

It doesn't necessarily determine **where** the messages are stored.

The implementation could store them:

```text
In memory
Redis
Firestore
Streamlit session state
Database
Custom storage
```

---

## Basic Concept

You can think of it as:

```python
class BaseChatMessageHistory:
    add_message(...)
    add_messages(...)
    messages
    clear()
```

The exact API can vary across LangChain versions, but the architectural idea remains the same.

---

# 9. ChatMessageHistory

`ChatMessageHistory` is a simple chat-history implementation that keeps messages in memory.

Typical usage:

```python
from langchain_core.chat_history import InMemoryChatMessageHistory

history = InMemoryChatMessageHistory()

history.add_user_message("My name is Arun")
history.add_ai_message("Nice to meet you, Arun")

print(history.messages)
```

Conceptually:

```text
Application
     ↓
InMemory Chat History
     ↓
RAM
```

---

## Example

```python
from langchain_core.chat_history import InMemoryChatMessageHistory

history = InMemoryChatMessageHistory()

history.add_user_message("My name is Arun")
history.add_ai_message("Nice to meet you!")

history.add_user_message("What is my name?")
```

History becomes approximately:

```text
Human: My name is Arun
AI: Nice to meet you!
Human: What is my name?
```

---

## Limitation

Because it is in memory:

```text
Application starts
       ↓
History exists

Application crashes/restarts
       ↓
History disappears
```

Therefore, it is useful for:

* learning
* testing
* prototypes
* short-lived sessions

but not necessarily ideal for persistent production chat history.

---

# 10. RedisChatMessageHistory

Redis is an **in-memory key-value database** commonly used for fast application state and caching.

`RedisChatMessageHistory` stores conversation history in Redis rather than only in Python memory.

Architecture:

```text
User
 ↓
Application
 ↓
RedisChatMessageHistory
 ↓
Redis
```

Suppose the user has:

```text
session_id = "user_123"
```

The application can associate:

```text
user_123
    ↓
Redis
    ↓
Conversation history
```

---

## Why Redis?

Redis provides:

* persistence options
* fast reads/writes
* shared state between application instances
* session-based storage
* scalability

This is particularly useful when your application has multiple backend instances.

```text
             ┌── Server 1
User ────────┼── Server 2
             └── Server 3
                    ↓
                  Redis
                    ↓
              Chat History
```

All servers can access the same history.

---

# 11. FirestoreChatMessageHistory

Firestore is a **cloud NoSQL database from Google Firebase**.

`FirestoreChatMessageHistory` allows chat messages to be stored in Firestore.

Architecture:

```text
User
 ↓
LangChain Application
 ↓
FirestoreChatMessageHistory
 ↓
Google Firestore
```

Example conceptual structure:

```text
Firestore
│
├── users
│     └── user_123
│           └── messages
│                 ├── message_1
│                 ├── message_2
│                 └── message_3
```

---

## When is it useful?

Firestore-based history is useful when:

* you're already using Firebase
* you need cloud persistence
* you want conversations to survive application restarts
* you need user/session-based storage

---

# 12. StreamlitChatMessageHistory

This is particularly useful when building **Streamlit chat applications**.

Streamlit applications commonly use:

```python
st.session_state
```

to maintain state during a user's session.

`StreamlitChatMessageHistory` integrates chat history with Streamlit's session state.

Architecture:

```text
User
 ↓
Streamlit App
 ↓
StreamlitChatMessageHistory
 ↓
st.session_state
```

Example conceptual usage:

```python
from langchain_community.chat_message_histories import (
    StreamlitChatMessageHistory
)

history = StreamlitChatMessageHistory(key="chat_history")
```

You can then add messages:

```python
history.add_user_message("Hello")
history.add_ai_message("Hi! How can I help?")
```

---

## Important limitation

Streamlit session state is generally **session-oriented**, not a replacement for a production persistent database.

For a production chatbot, you might instead use:

```text
Streamlit
   ↓
Backend API
   ↓
Redis / PostgreSQL / MongoDB / Firestore
```

---

# 13. Comparing the History Classes

| Class                         | Storage                 | Persistence               | Typical Use                 |
| ----------------------------- | ----------------------- | ------------------------- | --------------------------- |
| `BaseChatMessageHistory`      | Abstract                | Depends on implementation | Interface/base class        |
| `InMemoryChatMessageHistory`  | Python memory           | ❌                         | Testing/prototypes          |
| `RedisChatMessageHistory`     | Redis                   | ✅                         | Production/session storage  |
| `FirestoreChatMessageHistory` | Firestore               | ✅                         | Firebase/cloud applications |
| `StreamlitChatMessageHistory` | Streamlit session state | Session-based             | Streamlit applications      |

One correction worth noting: the commonly used LangChain class is **`InMemoryChatMessageHistory`**, not simply `ChatMessageHistory`, depending on the LangChain version/API you're using.

---

# 14. RunnableWithMessageHistory

This is the most important class in your list.

`RunnableWithMessageHistory` connects:

```text
Runnable
+
Chat History
+
Session ID
```

so that a LangChain runnable can automatically maintain conversational history.

---

## Why do we need it?

Without `RunnableWithMessageHistory`, you might manually do:

```python
history = get_history(session_id)

messages = history.messages

messages.append(
    HumanMessage(content=user_input)
)

response = llm.invoke(messages)

history.add_ai_message(response.content)
```

This becomes repetitive.

`RunnableWithMessageHistory` helps automate this workflow.

---

# 15. Architecture of RunnableWithMessageHistory

Think of it like this:

```text
                    session_id
                        │
                        ↓
User Message → RunnableWithMessageHistory
                        │
                        ↓
                get chat history
                        │
                        ↓
              Previous Messages
                        │
                        ↓
                 Current Message
                        │
                        ↓
                     Prompt
                        │
                        ↓
                      LLM
                        │
                        ↓
                    Response
                        │
                        ↓
                 Save to History
```

---

# 16. Simple Example

```python
from langchain_core.chat_history import InMemoryChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory
```

Create a history store:

```python
store = {}
```

Then create a function:

```python
def get_session_history(session_id):
    if session_id not in store:
        store[session_id] = InMemoryChatMessageHistory()

    return store[session_id]
```

Suppose you have a runnable:

```python
chain = prompt | llm
```

You can wrap it:

```python
chain_with_history = RunnableWithMessageHistory(
    chain,
    get_session_history,
    input_messages_key="input",
    history_messages_key="history",
)
```

Then invoke it:

```python
response = chain_with_history.invoke(
    {"input": "My name is Arun"},
    config={
        "configurable": {
            "session_id": "user_123"
        }
    }
)
```

The next call:

```python
response = chain_with_history.invoke(
    {"input": "What is my name?"},
    config={
        "configurable": {
            "session_id": "user_123"
        }
    }
)
```

can access the previous conversation.

---

# 17. Why Session ID Is Important

Imagine two users:

```text
User A → "My name is Arun"
User B → "My name is Rahul"
```

You don't want their histories mixed.

So:

```text
session_id = user_A
       ↓
History A

session_id = user_B
       ↓
History B
```

Architecture:

```text
                    RunnableWithMessageHistory
                              │
              ┌───────────────┴───────────────┐
              ↓                               ↓
        session_id=A                    session_id=B
              ↓                               ↓
         History A                       History B
              ↓                               ↓
      Arun's conversation            Rahul's conversation
```

This is fundamental for multi-user chat applications.

---

# 18. Complete Mental Model

You should remember the following architecture:

```text
                    USER
                     │
                     ↓
              Chat Application
                     │
                     ↓
          RunnableWithMessageHistory
                     │
                     ↓
              Session ID
                     │
                     ↓
             History Provider
                     │
       ┌─────────────┼─────────────┐
       ↓             ↓             ↓
    Memory         Redis       Firestore
       │
       ↓
 Chat Messages
       │
       ↓
 Human + AI + System Messages
       │
       ↓
     Prompt
       │
       ↓
      LLM
       │
       ↓
    Response
       │
       └──────────────→ Save response
```

---

# 19. Memory vs Chat History

These terms are related but not identical.

### Chat History

The actual messages:

```text
Human: Hello
AI: Hi
Human: What is Python?
AI: Python is...
```

### Memory

The broader mechanism that determines:

```text
What should be stored?
Where should it be stored?
How should it be retrieved?
What should be sent to the LLM?
```

Therefore:

```text
Chat History = Data

Memory = Mechanism / Strategy around that data
```

---

# 20. Does Storing Everything Mean Infinite Memory?

No.

A real chatbot cannot necessarily send the entire conversation forever because LLMs have a finite **context window**.

For example:

```text
Conversation
─────────────────────────────
Message 1
Message 2
Message 3
...
Message 5000
─────────────────────────────
             ↓
        Context Window
```

Eventually the conversation becomes too large.

Modern memory architectures therefore use techniques such as:

### 1. Conversation Buffer

Store all messages.

```text
M1 M2 M3 M4 M5 M6 ...
```

### 2. Conversation Summary

Summarize older messages.

```text
Old messages
     ↓
Summary
     +
Recent messages
```

### 3. Window Memory

Keep only the latest `N` messages.

```text
M1 M2 M3 M4 M5 M6 M7 M8

Keep:

M5 M6 M7 M8
```

### 4. Retrieval-Based Memory

Store information externally and retrieve relevant information when needed.

```text
Conversation
     ↓
Vector Database
     ↓
Relevant memories
     ↓
LLM
```

This starts to overlap with **RAG**.

---

# 21. Memory in Traditional LangChain vs Modern LangChain

You may see older tutorials containing classes such as:

```python
ConversationBufferMemory
ConversationSummaryMemory
ConversationBufferWindowMemory
```

These were heavily used in older LangChain APIs.

Modern LangChain commonly emphasizes:

```text
ChatMessageHistory
        +
RunnableWithMessageHistory
```

and more explicit state/history management.

So if you encounter an older tutorial, don't assume that its API is identical to current LangChain.

---

# 22. Interview Perspective

### Q: Does the LLM remember previous messages?

**Answer:**

> Not automatically. An LLM generates responses based on the input/context provided during an invocation. Conversational memory is implemented by the application, which stores previous messages and supplies relevant history back to the model.

---

### Q: What is `BaseChatMessageHistory`?

> It is the base abstraction for storing and managing chat messages. Different implementations can store messages in memory, Redis, Firestore, Streamlit session state, or another backend.

---

### Q: What is `RunnableWithMessageHistory`?

> It is a LangChain runnable wrapper that integrates a runnable with session-specific chat history, allowing previous messages to be automatically retrieved and updated across invocations.

---

### Q: Why is `session_id` important?

> It identifies a conversation/session and ensures that messages belonging to different users or conversations are kept separate.

---

### Q: Difference between Redis and in-memory history?

```text
InMemory
    ↓
Python process memory
    ↓
Fast
    ↓
Lost when process disappears

Redis
    ↓
External datastore
    ↓
Shared + persistent depending on configuration
    ↓
Better for production/session state
```

---

# 23. One-Line Revision Notes

| Concept                         | Remember This                                               |
| ------------------------------- | ----------------------------------------------------------- |
| **Memory**                      | Mechanism for retaining/reusing conversational information  |
| **LLM**                         | Doesn't automatically remember previous API calls           |
| **Chat History**                | Collection of previous chat messages                        |
| **BaseChatMessageHistory**      | Base abstraction for chat history                           |
| **InMemoryChatMessageHistory**  | Stores messages in application memory                       |
| **RedisChatMessageHistory**     | Stores messages in Redis                                    |
| **FirestoreChatMessageHistory** | Stores messages in Firestore                                |
| **StreamlitChatMessageHistory** | Stores history in Streamlit session state                   |
| **RunnableWithMessageHistory**  | Connects a runnable to session-based message history        |
| **Session ID**                  | Identifies a conversation/user session                      |
| **Context**                     | Information actually supplied to the LLM                    |
| **Memory ≠ Context**            | Memory retrieves/stores; context is what the model receives |

## The most important diagram to remember

```text
             LLM
              ↑
              │
          Context
              ↑
              │
        Chat History
              ↑
              │
           Memory
              ↑
              │
       Session / Database
```

And for LangChain:

```text
                    Runnable
                       │
                       ↓
          RunnableWithMessageHistory
                       │
                 session_id
                       │
                       ↓
          BaseChatMessageHistory
                       │
          ┌────────────┼────────────┐
          ↓            ↓            ↓
      InMemory        Redis      Firestore
          │
          ↓
      Messages
          │
          ↓
        Prompt
          │
          ↓
         LLM
```

**Core takeaway:** `BaseChatMessageHistory` defines how chat history is managed, concrete history classes determine **where/how it is stored**, and `RunnableWithMessageHistory` connects that history to a LangChain runnable so the conversation can persist across invocations.
